# The Ultimate Matchmaker: Trump's Birthday Fight Card 🎂

## Project Objective & Business Case

### 📌 Context
To celebrate Donald Trump's birthday, a special UFC fight card is being organized featuring exactly **7 matchups (14 fighters)**. As a massive fan of action-packed, high-drama fights, the goal is to make this birthday card the most entertaining event in UFC history.

### 🎯 Goal
The purpose of this project is to build an end-to-end data pipeline that:
1. Identifies the most exciting fighters in the UFC database using historical statistics.
2. Formulates an **"Audience Enjoyment" KPI (scaled 0 to 100)** to quantify how fan-friendly each fighter is.
3. Selects the **top 14 overall fighters** and groups them into **7 optimal fights** that maximize the combined enjoyment score for each match.

---

### 📊 The "Audience Enjoyment" Formula
To ensure maximum action, the KPI evaluates fighters based on four core pillars:
*   **Pace & Strike Volume (30% weight):** Fans love active striking. This measures total strikes landed + absorbed per minute.
*   **Knockdown Power (35% weight):** Heavy hitters who score knockdowns bring high-stakes drama.
*   **Finish Propensity (25% weight):** Fighters who finish fights (via KO or Submission) rather than going to a judge's decision.
*   **Dynamic Grappling (10% weight):** Active submission attempts instead of passive ground-and-pound/control time.

---

### 🛠️ Pipeline Architecture (Medallion Design)
We utilize a classic Delta Lakehouse architecture to move data from raw files to dashboard-ready insights:
*   **Bronze (`ufc_raw`):** Raw dataset ingested from Kagglehub with schema verification.
*   **Silver (`ufc_with_kpi`):** Cleansed and conformed data enriched with deterministic `fighter_id` keys and normalized component scores.
*   **Gold (`ufc_top_matchups` & `ufc_fighter_dashboard_metrics`):** Pre-aggregated matchups optimized for downstream visualization tools and dashboards.


###Methodology for the "Audience Enjoyment" KPI (0-100)
To build a realistic indicator of how exciting a fighter is, we combine four normalized components:

####Pace Score (30% weight): 
The sum of strikes landed and absorbed per minute. High output or high-damage wars are the most fan-friendly.

####Power Score (35% weight): 
Knockdown average. Heavy hitters who score knockdowns bring high drama.

####Finish Score (25% weight): 
KO and Submission win percentages combined. Fans love decisive finishes.

####Grappling Score (10% weight): 
Submission attempts average per 15 minutes. Active ground aggression is more engaging than control time.

Each component is normalized using Min-Max scaling (bringing the values between 0 and 100 relative to the rest of the dataset) to ensure the final KPI is perfectly bounded between 0 and 100

In [0]:
%pip install kagglehub

In [0]:
# Restart Python to ensure kagglehub is properly loaded
dbutils.library.restartPython()

In [0]:
# ============================================================================
# CONFIGURATION - Centralized settings for the entire pipeline
# ============================================================================

from dataclasses import dataclass
from typing import Dict
import os

@dataclass
class PipelineConfig:
    """Central configuration for UFC ETL pipeline"""
    
    # Catalog and Schema Configuration
    catalog: str = "workspace"
    bronze_schema: str = "bronze"
    silver_schema: str = "silver"
    gold_schema: str = "gold"
    
    # Table Names
    bronze_table: str = "ufc_raw"
    silver_table: str = "ufc_with_kpi"
    gold_matchups_table: str = "ufc_top_matchups"
    gold_metrics_table: str = "ufc_fighter_dashboard_metrics"
    
    # Data Source Configuration
    kaggle_dataset: str = "abhinavrana4140/ufc-fighters-database-2026"
    
    # KPI Weights (must sum to 1.0)
    kpi_weights: Dict[str, float] = None
    
    # Data Quality Thresholds
    min_required_rows: int = 10  # Minimum fighters required
    max_null_percentage: float = 0.3  # Max 30% nulls per column
    
    # Pipeline Behavior
    enable_schema_evolution: bool = True
    write_mode: str = "overwrite"  # Consider 'merge' for incremental in production
    
    def __post_init__(self):
        if self.kpi_weights is None:
            self.kpi_weights = {
                "pace": 0.30,
                "power": 0.35,
                "finish": 0.25,
                "grappling": 0.10
            }
        
        # Validate weights sum to 1.0
        weight_sum = sum(self.kpi_weights.values())
        if abs(weight_sum - 1.0) > 0.001:
            raise ValueError(f"KPI weights must sum to 1.0, got {weight_sum}")
    
    def get_table_name(self, layer: str, table: str) -> str:
        """Get fully qualified table name"""
        schema = getattr(self, f"{layer}_schema")
        return f"{self.catalog}.{schema}.{table}"

# Initialize configuration
config = PipelineConfig()

print("✅ Configuration loaded successfully")
print(f"Bronze: {config.get_table_name('bronze', config.bronze_table)}")
print(f"Silver: {config.get_table_name('silver', config.silver_table)}")
print(f"Gold Matchups: {config.get_table_name('gold', config.gold_matchups_table)}")
print(f"Gold Metrics: {config.get_table_name('gold', config.gold_metrics_table)}")

In [0]:
# ============================================================================
# UTILITY FUNCTIONS - Reusable helpers for pipeline operations
# ============================================================================

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from typing import List, Dict, Tuple
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def log_pipeline_step(step_name: str, status: str = "START"):
    """Log pipeline step with consistent formatting"""
    separator = "=" * 80
    logger.info(f"\n{separator}")
    logger.info(f"{status}: {step_name}")
    logger.info(separator)

def validate_data_quality(df: DataFrame, 
                         min_rows: int = 10,
                         max_null_pct: float = 0.3,
                         critical_columns: List[str] = None) -> Tuple[bool, List[str]]:
    """
    Validate data quality checks
    
    Returns:
        Tuple of (is_valid, list of issues)
    """
    issues = []
    
    # Check 1: Minimum row count
    row_count = df.count()
    if row_count < min_rows:
        issues.append(f"Insufficient data: {row_count} rows (minimum: {min_rows})")
    
    # Check 2: Null percentage per column
    total_rows = row_count
    for col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()
        null_pct = null_count / total_rows if total_rows > 0 else 0
        if null_pct > max_null_pct:
            issues.append(f"Column '{col}' has {null_pct:.1%} nulls (max: {max_null_pct:.1%})")
    
    # Check 3: Critical columns must exist and have no nulls
    if critical_columns:
        for col in critical_columns:
            if col not in df.columns:
                issues.append(f"Critical column missing: '{col}'")
            else:
                null_count = df.filter(F.col(col).isNull()).count()
                if null_count > 0:
                    issues.append(f"Critical column '{col}' has {null_count} null values")
    
    return len(issues) == 0, issues

def get_table_stats(df: DataFrame) -> Dict:
    """Get summary statistics for a DataFrame"""
    return {
        "row_count": df.count(),
        "column_count": len(df.columns),
        "columns": df.columns
    }

def ensure_schema_exists(catalog: str, schema: str):
    """Create schema if it doesn't exist (idempotent)"""
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
        logger.info(f"✅ Schema ensured: {catalog}.{schema}")
    except Exception as e:
        logger.error(f"❌ Failed to create schema {catalog}.{schema}: {str(e)}")
        raise

def write_delta_table(df: DataFrame, 
                     table_name: str, 
                     mode: str = "overwrite",
                     enable_schema_evolution: bool = True,
                     partition_by: List[str] = None):
    """
    Write DataFrame to Delta table with best practices
    
    Args:
        df: DataFrame to write
        table_name: Fully qualified table name
        mode: Write mode ('overwrite', 'append', 'merge')
        enable_schema_evolution: Allow schema changes
        partition_by: List of columns to partition by
    """
    try:
        writer = df.write.format("delta").mode(mode)
        
        if enable_schema_evolution:
            writer = writer.option("mergeSchema", "true")
        
        if partition_by:
            writer = writer.partitionBy(*partition_by)
        
        # Trigger the write action
        writer.saveAsTable(table_name)
        
        # Get stats after write (trigger action to confirm success)
        stats = spark.table(table_name).count()
        logger.info(f"✅ Successfully wrote {stats} rows to {table_name}")
        
    except Exception as e:
        logger.error(f"❌ Failed to write to {table_name}: {str(e)}")
        raise

print("✅ Utility functions loaded successfully")

In [0]:
# ============================================================================
# BRONZE LAYER - Raw Data Ingestion
# ============================================================================

log_pipeline_step("Bronze Layer - Data Ingestion", "START")

import os
import pandas as pd
import kagglehub
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# Define the expected Bronze schema
bronze_schema = StructType([
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("nickname", StringType(), True),
    StructField("sig_strikes_landed", DoubleType(), True),
    StructField("sig_strikes_attempted", DoubleType(), True),
    StructField("takedowns_landed", DoubleType(), True),
    StructField("takedowns_attempted", DoubleType(), True),
    StructField("str_landed_per_min", DoubleType(), True),
    StructField("str_absorbed_per_min", DoubleType(), True),
    StructField("td_avg_per_15min", DoubleType(), True),
    StructField("sub_avg_per_15min", DoubleType(), True),
    StructField("str_defense_pct", DoubleType(), True),
    StructField("td_defense_pct", DoubleType(), True),
    StructField("knockdown_avg", DoubleType(), True),
    StructField("striking_accuracy_pct", DoubleType(), True),
    StructField("takedown_accuracy_pct", DoubleType(), True),
    StructField("ko_wins", DoubleType(), True),
    StructField("ko_win_percentage", DoubleType(), True),
    StructField("dec_wins", DoubleType(), True),
    StructField("dec_win_percentage", DoubleType(), True),
    StructField("sub_wins", DoubleType(), True),
    StructField("sub_win_percentage", DoubleType(), True),
    StructField("avg_fight_time_minutes", DoubleType(), True)
])

# Step 1: Download & Load Raw File
try:
    logger.info(f"Downloading dataset from Kaggle: {config.kaggle_dataset}")
    dataset_path = kagglehub.dataset_download(config.kaggle_dataset)
    logger.info(f"Dataset downloaded to: {dataset_path}")
    
    # Locate the file
    data_files = [f for f in os.listdir(dataset_path) 
                  if f.endswith(('.csv', '.json', '.parquet', '.xlsx'))]
    
    if not data_files:
        raise FileNotFoundError(f"No supported data files found in {dataset_path}")
    
    file_path = data_files[0]
    full_path = os.path.join(dataset_path, file_path)
    logger.info(f"Loading file: {file_path}")
    
    # Load data into Pandas DataFrame
    if file_path.endswith('.csv'):
        pdf = pd.read_csv(full_path)
    elif file_path.endswith('.json'):
        pdf = pd.read_json(full_path)
    elif file_path.endswith('.parquet'):
        pdf = pd.read_parquet(full_path)
    elif file_path.endswith('.xlsx'):
        pdf = pd.read_excel(full_path)
    else:
        raise ValueError(f"Unsupported file type: {file_path}")
    
    logger.info(f"Loaded {len(pdf)} rows and {len(pdf.columns)} columns from Pandas")
    
except Exception as e:
    logger.error(f"Failed to download or load source data: {str(e)}")
    raise RuntimeError(f"Data ingestion failed: {str(e)}")

# Step 2: Validate Schema & Convert to PySpark DataFrame
try:
    expected_cols = [field.name for field in bronze_schema.fields]
    
    # Check for missing columns
    missing_cols = set(expected_cols) - set(pdf.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Select and align columns to match schema order
    pdf_aligned = pdf[expected_cols]
    
    # Convert to Spark DataFrame with schema enforcement
    df_spark = spark.createDataFrame(pdf_aligned, schema=bronze_schema)
    
    # IMPORTANT: Trigger action to validate the transformation worked
    row_count = df_spark.count()
    logger.info(f"Created Spark DataFrame with {row_count} rows")
    
except Exception as e:
    logger.error(f"Schema validation failed: {str(e)}")
    raise ValueError(f"Schema validation or conversion error: {str(e)}")

# Step 3: Data Quality Validation
logger.info("Running data quality checks...")
critical_cols = ["first", "last", "str_landed_per_min", "knockdown_avg"]
is_valid, issues = validate_data_quality(
    df_spark, 
    min_rows=config.min_required_rows,
    max_null_pct=config.max_null_percentage,
    critical_columns=critical_cols
)

if not is_valid:
    logger.warning(f"Data quality issues found: {len(issues)}")
    for issue in issues:
        logger.warning(f"  - {issue}")
    # In production, you might want to raise an exception or send alerts
    # For now, we'll continue with a warning
else:
    logger.info("✅ Data quality checks passed")

# Step 4: Write to Bronze Delta Table
try:
    bronze_table = config.get_table_name('bronze', config.bronze_table)
    ensure_schema_exists(config.catalog, config.bronze_schema)
    
    logger.info(f"Writing to Bronze table: {bronze_table}")
    write_delta_table(
        df_spark,
        bronze_table,
        mode=config.write_mode,
        enable_schema_evolution=config.enable_schema_evolution
    )
    
except Exception as e:
    logger.error(f"Failed to write Bronze table: {str(e)}")
    raise RuntimeError(f"Failed to write data to Delta table: {str(e)}")

# Display the resulting table
log_pipeline_step("Bronze Layer - Data Ingestion", "COMPLETE")
display(spark.table(bronze_table))


In [0]:
# ============================================================================
# SILVER LAYER - Data Enrichment with KPI Calculations
# ============================================================================

log_pipeline_step("Silver Layer - KPI Enrichment", "START")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# Define the target Silver schema (aligned with Bronze columns)
silver_schema = StructType([
    StructField("fighter_id", StringType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("nickname", StringType(), True),
    StructField("sig_strikes_landed", DoubleType(), True),
    StructField("sig_strikes_attempted", DoubleType(), True),
    StructField("takedowns_landed", DoubleType(), True),
    StructField("takedowns_attempted", DoubleType(), True),
    StructField("str_landed_per_min", DoubleType(), True),
    StructField("str_absorbed_per_min", DoubleType(), True),
    StructField("td_avg_per_15min", DoubleType(), True),
    StructField("sub_avg_per_15min", DoubleType(), True),
    StructField("str_defense_pct", DoubleType(), True),
    StructField("td_defense_pct", DoubleType(), True),
    StructField("knockdown_avg", DoubleType(), True),
    StructField("striking_accuracy_pct", DoubleType(), True),
    StructField("takedown_accuracy_pct", DoubleType(), True),
    StructField("ko_wins", DoubleType(), True),
    StructField("ko_win_percentage", DoubleType(), True),
    StructField("dec_wins", DoubleType(), True),
    StructField("dec_win_percentage", DoubleType(), True),
    StructField("sub_wins", DoubleType(), True),
    StructField("sub_win_percentage", DoubleType(), True),
    StructField("avg_fight_time_minutes", DoubleType(), True),
    StructField("pace_score", DoubleType(), True),
    StructField("power_score", DoubleType(), True),
    StructField("finish_score", DoubleType(), True),
    StructField("grappling_score", DoubleType(), True),
    StructField("audience_enjoyment_kpi", DoubleType(), True)
])

# Step 1: Load and Validate Bronze Data
bronze_table = config.get_table_name('bronze', config.bronze_table)
logger.info(f"Loading Bronze table: {bronze_table}")

try:
    bronze_data = spark.read.table(bronze_table)
    # Trigger action to validate read
    bronze_count = bronze_data.count()
    logger.info(f"Loaded {bronze_count} rows from Bronze")
except Exception as e:
    logger.error(f"Failed to load Bronze table: {str(e)}")
    raise RuntimeError(f"Failed to load Bronze source table: {str(e)}")

# Verify required fields exist
required_inputs = [
    "first", "last", "nickname", "str_landed_per_min", 
    "str_absorbed_per_min", "knockdown_avg", 
    "ko_win_percentage", "sub_win_percentage", "sub_avg_per_15min"
]

missing_inputs = set(required_inputs) - set(bronze_data.columns)
if missing_inputs:
    raise ValueError(f"Bronze table missing required fields: {missing_inputs}")


# Step 2: Perform Transformations and KPI Enrichment
logger.info("Calculating KPI components...")

# Generate unique ID and raw components
df_with_raw = bronze_data.withColumn(
        "fighter_id",
        F.md5(F.concat_ws("_", 
            F.coalesce(F.col("first"), F.lit("")), 
            F.coalesce(F.col("last"), F.lit("")), 
            F.coalesce(F.col("nickname"), F.lit(""))
        ))
    ).withColumn(
        "pace_raw",
        (F.coalesce("str_landed_per_min", F.lit(0.0)) + F.coalesce("str_absorbed_per_min", F.lit(0.0)))
    ).withColumn(
        "power_raw",
        F.coalesce("knockdown_avg", F.lit(0.0))
    ).withColumn(
        "finish_raw",
        (F.coalesce("ko_win_percentage", F.lit(0.0)) + F.coalesce("sub_win_percentage", F.lit(0.0)))
    ).withColumn(
        "grappling_raw",
        F.coalesce("sub_avg_per_15min", F.lit(0.0))
    )

# Define global window for normalization
global_window = Window.partitionBy()

# Add normalized components (0-100) using configured weights
logger.info("Normalizing KPI scores (0-100 scale)...")
df_scores = df_with_raw.withColumn(
        "pace_score",
        F.round(
            F.when(F.max("pace_raw").over(global_window) == F.min("pace_raw").over(global_window), F.lit(0.0))
            .otherwise((F.col("pace_raw") - F.min("pace_raw").over(global_window)) / 
                       (F.max("pace_raw").over(global_window) - F.min("pace_raw").over(global_window)) * 100), 
            2
        )
    ).withColumn(
        "power_score",
        F.round(
            F.when(F.max("power_raw").over(global_window) == F.min("power_raw").over(global_window), F.lit(0.0))
            .otherwise((F.col("power_raw") - F.min("power_raw").over(global_window)) / 
                       (F.max("power_raw").over(global_window) - F.min("power_raw").over(global_window)) * 100), 
            2
        )
    ).withColumn(
        "finish_score",
        F.round(
            F.when(F.max("finish_raw").over(global_window) == F.min("finish_raw").over(global_window), F.lit(0.0))
            .otherwise((F.col("finish_raw") - F.min("finish_raw").over(global_window)) / 
                       (F.max("finish_raw").over(global_window) - F.min("finish_raw").over(global_window)) * 100), 
            2
        )
    ).withColumn(
        "grappling_score",
        F.round(
            F.when(F.max("grappling_raw").over(global_window) == F.min("grappling_raw").over(global_window), F.lit(0.0))
            .otherwise((F.col("grappling_raw") - F.min("grappling_raw").over(global_window)) / 
                       (F.max("grappling_raw").over(global_window) - F.min("grappling_raw").over(global_window)) * 100), 
            2
        )
    )

# Calculate final KPI using configured weights, sort, and drop temporary columns
logger.info("Computing final Audience Enjoyment KPI...")
silver_ufc_df = df_scores.withColumn(
    "audience_enjoyment_kpi",
    F.round(
        (F.col("pace_score") * config.kpi_weights["pace"]) + 
        (F.col("power_score") * config.kpi_weights["power"]) + 
        (F.col("finish_score") * config.kpi_weights["finish"]) + 
        (F.col("grappling_score") * config.kpi_weights["grappling"]),
        2
    )
).drop("pace_raw", "power_raw", "finish_raw", "grappling_raw") \
 .orderBy(F.desc("audience_enjoyment_kpi"))


# Step 3: Enforce Silver Schema Alignment
logger.info("Aligning to Silver schema...")
try:
    schema_cols = [F.col(field.name).cast(field.dataType).alias(field.name) 
                   for field in silver_schema.fields]
    silver_aligned_df = silver_ufc_df.select(*schema_cols)
    # Trigger action to validate schema alignment
    aligned_count = silver_aligned_df.count()
    logger.info(f"Schema alignment successful: {aligned_count} rows")
except Exception as e:
    logger.error(f"Schema alignment failed: {str(e)}")
    raise TypeError(f"Schema alignment or casting failed: {str(e)}")


# Step 4: Write to Silver Delta Table
silver_table = config.get_table_name('silver', config.silver_table)
ensure_schema_exists(config.catalog, config.silver_schema)

logger.info(f"Writing to Silver table: {silver_table}")
try:
    write_delta_table(
        silver_aligned_df,
        silver_table,
        mode=config.write_mode,
        enable_schema_evolution=config.enable_schema_evolution
    )
except Exception as e:
    logger.error(f"Failed to write Silver table: {str(e)}")
    raise RuntimeError(f"Failed to write Silver Delta table: {str(e)}")

# Display the final output
log_pipeline_step("Silver Layer - KPI Enrichment", "COMPLETE")
display(spark.table(silver_table))


In [0]:
# ============================================================================
# GOLD LAYER - Business Aggregations & Fight Matchups
# ============================================================================

log_pipeline_step("Gold Layer - Aggregations & Matchups", "START")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType, BooleanType

# Define the Gold Matchups Schema
gold_matchups_schema = StructType([
    StructField("fight_id", IntegerType(), True),
    StructField("f1_id", StringType(), True),
    StructField("f1_name", StringType(), True),
    StructField("f1_nickname", StringType(), True),
    StructField("f1_kpi", DoubleType(), True),
    StructField("f2_id", StringType(), True),
    StructField("f2_name", StringType(), True),
    StructField("f2_nickname", StringType(), True),
    StructField("f2_kpi", DoubleType(), True),
    StructField("combined_kpi", DoubleType(), True)
])

# 2. Define the Gold Fighter Dashboard Schema (Silver + helper columns, aligned with available data)
gold_metrics_schema = StructType([
    StructField("fighter_id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("is_top_14", BooleanType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("nickname", StringType(), True),
    StructField("sig_strikes_landed", DoubleType(), True),
    StructField("sig_strikes_attempted", DoubleType(), True),
    StructField("takedowns_landed", DoubleType(), True),
    StructField("takedowns_attempted", DoubleType(), True),
    StructField("str_landed_per_min", DoubleType(), True),
    StructField("str_absorbed_per_min", DoubleType(), True),
    StructField("td_avg_per_15min", DoubleType(), True),
    StructField("sub_avg_per_15min", DoubleType(), True),
    StructField("str_defense_pct", DoubleType(), True),
    StructField("td_defense_pct", DoubleType(), True),
    StructField("knockdown_avg", DoubleType(), True),
    StructField("striking_accuracy_pct", DoubleType(), True),
    StructField("takedown_accuracy_pct", DoubleType(), True),
    StructField("ko_wins", DoubleType(), True),
    StructField("ko_win_percentage", DoubleType(), True),
    StructField("dec_wins", DoubleType(), True),
    StructField("dec_win_percentage", DoubleType(), True),
    StructField("sub_wins", DoubleType(), True),
    StructField("sub_win_percentage", DoubleType(), True),
    StructField("avg_fight_time_minutes", DoubleType(), True),
    StructField("pace_score", DoubleType(), True),
    StructField("power_score", DoubleType(), True),
    StructField("finish_score", DoubleType(), True),
    StructField("grappling_score", DoubleType(), True),
    StructField("audience_enjoyment_kpi", DoubleType(), True)
])


# Step 1: Load Silver Data
silver_table = config.get_table_name('silver', config.silver_table)
logger.info(f"Loading Silver table: {silver_table}")

try:
    silver_df = spark.read.table(silver_table)
    # Trigger action to validate read
    silver_count = silver_df.count()
    logger.info(f"Loaded {silver_count} fighters from Silver")
except Exception as e:
    logger.error(f"Failed to load Silver table: {str(e)}")
    raise RuntimeError(f"Failed to load Silver table: {str(e)}")


# Step 2: Transform and Build Gold Fighter Metrics Table
logger.info("Ranking fighters and enriching with dashboard metrics...")

# Rank all fighters by KPI to identify the top 14
window_spec = Window.orderBy(F.desc("audience_enjoyment_kpi"))
df_ranked = silver_df.withColumn("rank", F.row_number().over(window_spec))

# Enrich with dashboard convenience columns
df_enriched = df_ranked.withColumn(
    "full_name", 
    F.concat_ws(" ", F.col("first"), F.col("last"))
).withColumn(
    "is_top_14", 
    F.col("rank") <= 14
)

# Enforce Gold Metrics schema casting and ordering
try:
    metrics_cols = [F.col(f.name).cast(f.dataType).alias(f.name) 
                    for f in gold_metrics_schema.fields]
    gold_metrics_df = df_enriched.select(*metrics_cols)
    # Trigger action to validate transformation
    metrics_count = gold_metrics_df.count()
    logger.info(f"Created Gold metrics DataFrame with {metrics_count} fighters")
except Exception as e:
    logger.error(f"Gold Fighter Metrics transformation failed: {str(e)}")
    raise RuntimeError(f"Failed during Gold Fighter Metrics transformation: {str(e)}")


# Step 3: Match the Top 14 Fighters into 7 Optimal Fights
logger.info("Generating fight matchups for top 14 fighters...")

# Filter for the top 14 and assign a fight ID
top_14_fights = df_enriched.filter(F.col("is_top_14") == True) \
    .withColumn("fight_id", (F.floor((F.col("rank") - 1) / 2) + 1).cast("int")) \
    .withColumn("fighter_position", 
                F.when(F.col("rank") % 2 == 1, "fighter_1").otherwise("fighter_2"))

# Separate into Fighter 1 and Fighter 2 roles
f1_df = top_14_fights.filter(F.col("fighter_position") == "fighter_1").select(
    F.col("fight_id"),
    F.col("fighter_id").alias("f1_id"),
    F.col("full_name").alias("f1_name"),
    F.col("nickname").alias("f1_nickname"),
    F.col("audience_enjoyment_kpi").alias("f1_kpi")
)

f2_df = top_14_fights.filter(F.col("fighter_position") == "fighter_2").select(
    F.col("fight_id"),
    F.col("fighter_id").alias("f2_id"),
    F.col("full_name").alias("f2_name"),
    F.col("nickname").alias("f2_nickname"),
    F.col("audience_enjoyment_kpi").alias("f2_kpi")
)

# Join the pairs and calculate combined KPI
matchups_df = f1_df.join(f2_df, "fight_id") \
    .withColumn("combined_kpi", F.round(F.col("f1_kpi") + F.col("f2_kpi"), 2))

# Enforce Gold Matchups schema casting and ordering
try:
    matchup_cols = [F.col(f.name).cast(f.dataType).alias(f.name) 
                    for f in gold_matchups_schema.fields]
    gold_matchups_df = matchups_df.select(*matchup_cols).orderBy("fight_id")
    # Trigger action to validate matchup generation
    matchup_count = gold_matchups_df.count()
    logger.info(f"Created {matchup_count} fight matchups")
except Exception as e:
    logger.error(f"Matchup generation failed: {str(e)}")
    raise RuntimeError(f"Failed during Matchup generation: {str(e)}")


# Step 4: Write Gold Tables to Delta
metrics_table = config.get_table_name('gold', config.gold_metrics_table)
matchups_table = config.get_table_name('gold', config.gold_matchups_table)

ensure_schema_exists(config.catalog, config.gold_schema)

try:
    # Write dashboard metrics table
    logger.info(f"Writing to Gold metrics table: {metrics_table}")
    write_delta_table(
        gold_metrics_df,
        metrics_table,
        mode=config.write_mode,
        enable_schema_evolution=config.enable_schema_evolution
    )
    
    # Write matchups table
    logger.info(f"Writing to Gold matchups table: {matchups_table}")
    write_delta_table(
        gold_matchups_df,
        matchups_table,
        mode=config.write_mode,
        enable_schema_evolution=config.enable_schema_evolution
    )
    
except Exception as e:
    logger.error(f"Failed to write Gold tables: {str(e)}")
    raise RuntimeError(f"Failed writing tables to Gold layer: {str(e)}")

# Display the 7 matchups
log_pipeline_step("Gold Layer - Aggregations & Matchups", "COMPLETE")
logger.info(f"✅ Pipeline completed successfully!")
logger.info(f"  - Bronze: {config.get_table_name('bronze', config.bronze_table)}")
logger.info(f"  - Silver: {config.get_table_name('silver', config.silver_table)}")
logger.info(f"  - Gold Metrics: {metrics_table}")
logger.info(f"  - Gold Matchups: {matchups_table}")

display(spark.table(matchups_table))
